# M3 R1 robustness: does the model look at the snake or at the scenery?

We take one trained run's `best.pt`, with no further training, and rescore every validation photo four ways to answer
R1: does the model name the snake, or the scenery around it? We run the notebook once for the F0 run and once for the
FE winner.

- **(a) As is:** the input the run was validated on.
- **(b) Snake blanked:** the snake's mask pixels set to the ImageNet mean grey (0 after normalisation), every other
  pixel unchanged. The **shortcut rate** is the share of photos right as they are that stay right without the snake.
- **(c) Background swapped:** the snake's pixels pasted, unchanged, on the scenery of a validation photo of another
  class (a seeded draw; the donor's own snake is blanked first, then the donor is resized to this photo). We report how
  often the prediction keeps its own class and how often it follows the donor's, against the control of how often it
  already named the donor's class before the swap. The **silhouette control** costs no GPU time: how often the donor's
  own blanked photo is called the donor's class, and follows-donor again without those swaps.
- **(d) Slices:** Indian against non-Indian photos, dark (notebook 06's exposure rule), blurry (Laplacian variance
  below `BLUR_BELOW`), and classes with 10 or fewer training photographers (29 classes, 119 validation photos; a
  cut-off of 3 left the slice empty), each with photographer-grouped bootstrap intervals.

**Masks.** Validation photos have no SAM masks, so (b) and (c) use the front end's **predicted** masks
(`frontend_preds.csv`; the metrics record the segmenter that wrote them, from the predictions' `summary.json`). Photos
without a usable predicted mask of the photo's size are left out of (b) and (c) and counted.

**Caveats** for the shortcut rate and follows-donor: a predicted mask can miss part of the snake (a tail tip, a
coil), and those pixels survive blanking, so both are upper bounds on reliance on scenery; and a blanked snake leaves a
grey silhouette with its outline, which a model can read as shape. The silhouette control measures the second for (c).

**Verdicts.** The verdict-level effect reads `decision.decide` with the thresholds `decision_report` fits on the run's
own calibration outputs, with the same `RECALL_BOUND` as the run's decision file. A by-product, `quality.csv` (path,
set, dark, blurry, brightness, Laplacian variance) for the validation, calibration and unknown-pool half A photos, is
the quality table `decision_report(quality=...)` reads.

**Inputs:** `serpentguard-indian-snakes-clean` and `serpentguard-m3-subset`, found with `kagglehub.dataset_download`.
The run and the predictions come from other kernels' outputs through `kagglehub.notebook_output_download`, which a
batch kernel can read only when they are declared as kernel sources: `RUN_KERNEL`, the training kernel, whose output
holds the folder `RUN_ID` (`best.pt`, `config.yaml`, `calib_outputs.npz`, `prototypes.npz`, `val_outputs.npz`), and
`PREDS_KERNEL`, the notebook 13 kernel, whose output holds `frontend_preds.csv` and `masks_pred/`. Either handle can
instead be `"dataset:owner/slug"`, a private dataset copy. The R1 code is in this notebook (section 0);
the rest comes from `../src`, which the runner kernel unpacks into `/kaggle/working/repo`.

**Outputs:** `/kaggle/working/robustness/quality.csv` and `/kaggle/working/robustness/<RUN_ID>/`, which holds
`robustness_metrics.json`, `summary.csv` and each condition's per-photo outputs, verdicts included, embeddings as
float16.

**Run:** on Kaggle with one GPU. `FEW_PHOTOGRAPHERS` changes the slice's cut-off, `RECALL_BOUND` picks the decision
file's fit, and `LIMIT` makes a smoke run.

**Unattended runs.** The subset is checked before any GPU time; the as-is pass must reproduce the run's own saved
validation top-1 (`INSTRUMENT_FLOOR`) before (b) and (c) start; each condition's outputs are saved as its pass ends.

In [1]:
RUN_KERNEL = 'dataset:puneetbajajiitm/serpentguard-m3-run-final'
RUN_ID = 'H1_top1_s1'
PREDS_KERNEL = 'puneetbajajiitm/sg-m3-frontends'
FEW_PHOTOGRAPHERS = None  # None keeps the module's cut-off (10: 29 classes, 119 validation photos)
LIMIT = None              # first N photos of each set, for a smoke run
RECALL_BOUND = 'lower'

In [2]:
import os
import subprocess
import sys

ALBUMENTATIONS_VERSION = "2.0.8"  # requirements-m3.txt; the evaluation transform the runs were scored with
TIMM_VERSION = "1.0.31"            # requirements-m3.txt; the encoders' weights load into it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"albumentations=={ALBUMENTATIONS_VERSION}",
                f"timm=={TIMM_VERSION}"], check=True)
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"  # otherwise albumentations asks PyPI for its latest version on import

import albumentations
import timm

assert albumentations.__version__ == ALBUMENTATIONS_VERSION, f"albumentations {albumentations.__version__} is loaded"
assert timm.__version__ == TIMM_VERSION, f"timm {timm.__version__} is loaded"
print(f"albumentations {albumentations.__version__}, timm {timm.__version__}")

albumentations 2.0.8, timm 1.0.31


In [3]:
SCIKIT_LEARN_VERSION = "1.9.1"  # requirements-m3.txt; every metric uses it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SCIKIT_LEARN_VERSION}"], check=True)

import sklearn

assert sklearn.__version__ == SCIKIT_LEARN_VERSION, (f"scikit-learn {sklearn.__version__} is loaded; restart the "
                                                     f"kernel after the install")
print(f"scikit-learn {sklearn.__version__}")

scikit-learn 1.9.1


In [4]:
import json
import time
from pathlib import Path

import kagglehub

CODE = Path("/kaggle/input/serpentguard-code/src")
CODE = CODE if CODE.exists() else Path("../src").resolve()
sys.path.insert(0, str(CODE))

import pandas as pd
import torch

from serpentguard.config import load_config
from serpentguard.run_data import FRONTEND_PREDS_FILE
from serpentguard.run_outputs import subset_hash
from serpentguard.train import (CALIBRATION_OUTPUTS, CHECKPOINT_FILE, CONFIG_FILE, HASH_FILE, PROTOTYPES_FILE,
                                VAL_OUTPUTS)

OWNER = "puneetbajajiitm"


def input_root(slug):
    """Where an attached dataset is mounted; Kaggle no longer mounts datasets at /kaggle/input/<slug>."""
    return Path(kagglehub.dataset_download(f"{OWNER}/{slug}"))


DATASET_PREFIX = "dataset:"  # "dataset:owner/slug": a private dataset copy of a kernel's output, for another account


def kernel_output(handle):
    """A kernel's output folder, or for "dataset:owner/slug" a private dataset copy of it."""
    if handle.startswith(DATASET_PREFIX):
        return Path(kagglehub.dataset_download(handle.removeprefix(DATASET_PREFIX)))
    return Path(kagglehub.notebook_output_download(handle))


DATA = input_root("serpentguard-indian-snakes-clean")
SUBSET = input_root("serpentguard-m3-subset") / "subset.csv"
RUN_OUTPUT = kernel_output(RUN_KERNEL)
runs = sorted(path.parent for path in RUN_OUTPUT.rglob(CHECKPOINT_FILE) if path.parent.name == RUN_ID)
assert len(runs) == 1, f"{RUN_KERNEL}'s output must hold one {RUN_ID} folder with {CHECKPOINT_FILE}; found {runs}"
RUN = runs[0]
preds_files = sorted(kernel_output(PREDS_KERNEL).rglob(FRONTEND_PREDS_FILE))
assert len(preds_files) == 1, f"{PREDS_KERNEL}'s output must hold one {FRONTEND_PREDS_FILE}; found {preds_files}"
PREDS = preds_files[0].parent
OUT = Path("/kaggle/working/robustness")
EXPECTED_SUBSET_HASH = "40196d19a68e"  # first 12 hex digits of the real subset's SHA-256

for required in (DATA / "metadata.csv", DATA / "classes.csv", DATA / "data", SUBSET, PREDS / "masks_pred",
                 *(RUN / name for name in (CONFIG_FILE, HASH_FILE, CALIBRATION_OUTPUTS, PROTOTYPES_FILE, VAL_OUTPUTS))):
    assert required.exists(), f"missing input {required}"
cfg = load_config(RUN / CONFIG_FILE)
assert cfg.run_id == RUN_ID, f"{RUN} holds run {cfg.run_id}"
subset_digest = subset_hash(pd.read_csv(SUBSET, usecols=["path"])["path"])  # before any GPU time
assert subset_digest.startswith(EXPECTED_SUBSET_HASH), f"{SUBSET} is another subset ({subset_digest[:12]})"
assert (RUN / HASH_FILE).read_text().strip() == subset_digest, f"{RUN_ID} was trained on another subset"
assert torch.cuda.is_available(), "no GPU in this session"
workers = os.cpu_count() or 2
OUT.mkdir(parents=True, exist_ok=True)
print(f"code {CODE}; {torch.cuda.device_count()} GPUs, {workers} CPUs; run {RUN_ID} ({cfg.front_end}) at {RUN}; "
      f"predictions {PREDS}")

code /kaggle/working/repo/src; 2 GPUs, 4 CPUs; run H1_top1_s1 (F1) at /kaggle/input/datasets/puneetbajajiitm/serpentguard-m3-run-final/H1_top1_s1; predictions /kaggle/input/notebooks/puneetbajajiitm/sg-m3-frontends/frontend_preds


## 0. The R1 code

The conditions, the quality table, the summaries and `score_run`, which writes `robustness_metrics.json` and each
condition's per-photo outputs. Every summary is computed from those per-photo outputs.

In [5]:
import json
import math
import os
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict
from pathlib import Path

os.environ.setdefault("YOLO_OFFLINE", "1")  # read once by ultralytics on import: no pip installs, no analytics

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image, UnidentifiedImageError
from torch.utils.data import DataLoader, Dataset

from serpentguard.data import (AUGMENT, BOX, DARK_BELOW, FRONT_END_MODES, ID_TYPES, IMAGE_FOLDER, IMAGENET_MEAN, SEED,
                               UNKNOWN_POOL, apply_mode, build_frame, cap_train, load_mask, load_rgb, with_files)
from serpentguard.decision import MEDICAL_CSV, POINT, VERDICTS, danger_tables, decide, fit_thresholds
from serpentguard.frontend import S1, SUMMARY_FILE
from serpentguard.metrics import assert_no_test_paths, grouped_bootstrap, species_report
from serpentguard.run_data import (CLASSES_FILE, EVAL_BATCH_SCALE, FRONTEND_PREDS_FILE, MASK_INPUT_MODES, METADATA_FILE,
                                   OPENCV_THREADS_PER_WORKER, PREDICTED, Tables, warm_page_cache)
from serpentguard.run_files import CALIBRATION_OUTPUTS, CHECKPOINT_FILE, PROTOTYPES_FILE, VAL_OUTPUTS
from serpentguard.run_outputs import (class_tables, file_sha256, library_versions, load_checkpoint, photographer_groups,
                                      predict, save_outputs, softmax, subset_hash, write_json)
from serpentguard.subset import calibration_set, unknown_split

BLANK_RGB = tuple(round(255 * mean) for mean in IMAGENET_MEAN)  # the ImageNet mean pixel: 0 after normalisation
BLUR_BELOW = 100.0          # variance of the grey photo's Laplacian, at its stored size (500 px long side for 99.6%)
SWAP_SEED = SEED
NO_DONOR = NO_PREDICTION = -1
FEW_PHOTOGRAPHERS_CUTOFF = 10  # at most this many training photographers (29 classes, 119 validation photos)
INSTRUMENT_FLOOR = 0.99     # the as-is pass must give the run's own saved top-1 on this share of photos (AMP rounding)
SLICE_COLUMNS = {"india": "in_india", "dark": "dark", "blurry": "blurry", "few_photographers": "few_photographers"}
SLICES = tuple(SLICE_COLUMNS)
AS_IS, BLANKED, SWAPPED = "as_is", "blanked", "swapped"
CONDITIONS = (AS_IS, BLANKED, SWAPPED)
OUTPUT_FILES = {name: f"robustness_{name}.npz" for name in CONDITIONS}
QUALITY_FILE, METRICS_FILE = "quality.csv", "robustness_metrics.json"
QUALITY_COLUMNS = ["path", "set", "dark", "blurry", "brightness", "laplacian_var"]
QUALITY_WORKERS = 8         # threads reading photos or masks
MASK_USABLE, MASK_NONE, MASK_DROPPED = "usable", "none", "dropped_by_cleaning"
MASK_MISFIT, MASK_UNREADABLE = "size_mismatch", "unreadable"
# offline, ultralytics' patched PIL.Image.open raises ModuleNotFoundError for a file Pillow can't identify
PHOTO_DECODE_ERRORS = (UnidentifiedImageError, ImportError)
ERROR_EXAMPLES = 5
SCORE_KEYS = ("top1", "top3", "macro_f1", "balanced_acc")
PROTOTYPE_SET = "plain"
NOT_AVAILABLE = "not available"
NAN_INTERVAL = [math.nan, math.nan]

MASKS_NOTE = ("(b) and (c) cut the snake with the front end's predicted masks (frontend_preds.csv; the segmenter is in "
              "provenance): validation photos have no SAM masks; photos without a usable predicted mask of the "
              "photo's size are left out of both")
DONOR_NOTE = ("another validation photo with a predicted mask, of a different class, drawn with the seed; its own "
              "snake is blanked first, so only its scenery goes behind this photo's snake, resized to this photo")
VERDICT_NOTE = ("decision.decide with fit_thresholds on the run's calibration outputs (decision_report's thresholds, "
                "with the same recall_bound) and the plain prototypes; no front-end or quality flags, so only the "
                "image changes")
SHORTCUT_NOTE = "shortcut_rate: of the photos right as they are, the share still right with the snake blanked"
MASK_MISS_NOTE = ("predicted masks can miss part of the snake (a tail tip, a coil), and those pixels survive blanking "
                  "and move with the snake: shortcut_rate and follows_donor are upper bounds on reliance on scenery")
SHORTCUT_SILHOUETTE_NOTE = ("the blanked snake leaves a grey silhouette with its outline, so a photo still right with "
                            "the snake blanked may be read from that shape rather than the scenery")
DONOR_SILHOUETTE_NOTE = ("the donor's own snake is blanked too and its grey silhouette stays in the scenery, so a swap "
                         "can follow the donor's shape rather than its scenery: donor_silhouette_rate is how often the "
                         "donor's blanked photo alone is called the donor's class, and "
                         "follows_donor_without_silhouette leaves those swaps out")
BOOTSTRAP_NOTE = "95% intervals from 1,000 photographer-grouped resamples"


def blank_snake(img, mask, fill=BLANK_RGB):
    """img (H x W x 3 uint8) with its mask pixels set to fill."""
    blanked = img.copy()
    blanked[np.asarray(mask).astype(bool)] = fill
    return blanked


def swap_background(img, mask, background):
    """img's snake (its mask pixels, unchanged) on background, resized to img's size."""
    height, width = img.shape[:2]
    scenery = cv2.resize(np.ascontiguousarray(background), (width, height), interpolation=cv2.INTER_LINEAR)
    return np.where(np.asarray(mask).astype(bool)[..., None], img, scenery).astype(np.uint8)


def draw_donors(class_ids, eligible, seed=SWAP_SEED):
    """Per photo, the row of a seeded eligible photo of another class; NO_DONOR when it isn't eligible or none is."""
    class_ids, eligible = np.asarray(class_ids), np.asarray(eligible, dtype=bool)
    pool = np.flatnonzero(eligible)
    order = pool[np.argsort(class_ids[pool], kind="stable")]
    ranked = class_ids[order]
    start = np.searchsorted(ranked, class_ids[pool], side="left")
    end = np.searchsorted(ranked, class_ids[pool], side="right")
    others = len(pool) - (end - start)
    draw = np.floor(np.random.default_rng(seed).random(len(pool)) * others).astype(int)
    position = np.where(draw < start, draw, draw + end - start)  # skip the photo's own class block
    donors = np.full(len(class_ids), NO_DONOR)
    has_other = others > 0
    donors[pool[has_other]] = order[position[has_other]]
    return donors


def load_raw_rgb(path):
    """The photo as stored, before notebook 06's exposure fix."""
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB"), dtype=np.uint8)


def image_quality(rgb):
    """Notebook 06's dark rule and the Laplacian blur rule, on a raw photo."""
    brightness = float(rgb.mean())
    laplacian = float(cv2.Laplacian(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var())
    return {"dark": brightness < DARK_BELOW, "blurry": laplacian < BLUR_BELOW, "brightness": brightness,
            "laplacian_var": laplacian}


def quality_frames(metadata, subset):
    """The photos quality.csv covers: validation, the calibration set and unknown-pool half A."""
    train = metadata[metadata["split"] == "train"]
    frames = {"val": metadata[metadata["split"] == "val"], "calibration": calibration_set(train, subset),
              "unknown": unknown_split(metadata[metadata["split"] == UNKNOWN_POOL])[0]}
    for frame in frames.values():
        assert_no_test_paths(frame)
    return {name: frame[["path"]].reset_index(drop=True) for name, frame in frames.items()}


def quality_table(frames, data_root, workers=QUALITY_WORKERS):
    """QUALITY_COLUMNS for every photo of frames ({set name: frame with path})."""
    rows = pd.concat([frame[["path"]].assign(set=name) for name, frame in frames.items()], ignore_index=True)
    files = [Path(data_root, IMAGE_FOLDER, path) for path in rows["path"]]
    with ThreadPoolExecutor(max_workers=workers) as pool:
        measured = list(pool.map(lambda file: image_quality(load_raw_rgb(file)), files))
    return pd.concat([rows, pd.DataFrame(measured, columns=QUALITY_COLUMNS[2:])], axis=1)[QUALITY_COLUMNS]


def quality_summary(quality):
    return {name: {"photos": len(rows), "dark": share(flags(rows["dark"])), "blurry": share(flags(rows["blurry"]))}
            for name, rows in quality.groupby("set", sort=False)}

In [6]:
def few_photographer_classes(train, n_classes, most=FEW_PHOTOGRAPHERS_CUTOFF):
    """Per class: True when the training frame holds at most `most` photographers of it."""
    counts = pd.Series(photographer_groups(train)).groupby(train["class_id"].to_numpy()).nunique()
    photographers = np.zeros(n_classes, dtype=int)
    photographers[counts.index.to_numpy(dtype=int)] = counts.to_numpy()
    return photographers <= most


def slice_metrics(frame, probs, slices=SLICES, *, train_counts, tier_of_class, val_counts=None):
    """Each slice's photos against the rest, with the top-1 gap and its interval; val_counts default to the frame's
    own, which must then be the full validation set."""
    labels = frame["class_id"].to_numpy()
    val_counts = np.bincount(labels, minlength=probs.shape[1]) if val_counts is None else val_counts
    groups = photographer_groups(frame)
    right = probs.argmax(axis=1) == labels
    report = {}
    for name in slices:
        inside = flags(frame[SLICE_COLUMNS[name]])
        sides = {side: scores(labels[members], probs[members], groups[members], train_counts, val_counts, tier_of_class)
                 for side, members in (("in", inside), ("out", ~inside))}
        report[name] = {**sides, "top1_gap": sides["in"]["top1"] - sides["out"]["top1"], "top1_gap_ci": interval(
            lambda rows, inside=inside: share(right[rows][inside[rows]]) - share(right[rows][~inside[rows]]), groups)}
    return report


def scores(labels, probs, groups, train_counts, val_counts, tier_of_class):
    if not len(labels):
        return {"photos": 0, "photographers": 0, **dict.fromkeys(SCORE_KEYS, math.nan), "top1_ci": NAN_INTERVAL}
    report = species_report(labels, probs, train_counts, val_counts, tier_of_class)
    right = probs.argmax(axis=1) == labels
    return {"photos": len(labels), "photographers": int(np.unique(groups).size),
            **{key: report[key] for key in SCORE_KEYS}, "top1_ci": interval(lambda rows: share(right[rows]), groups)}


def shortcut_report(labels, as_is, blanked, groups):
    """Top-1 predictions as they are and with the snake blanked."""
    right, still = as_is == labels, blanked == labels
    return {"right_as_is": int(right.sum()), "shortcut_rate": share(still[right]),
            "shortcut_rate_ci": interval(lambda rows: share(still[rows][right[rows]]), groups),
            "blanked_top1": share(still), "changed_prediction": share(blanked != as_is)}


def donor_report(labels, donor_labels, as_is, swapped, groups, donor_blanked):
    """Top-1 predictions as they are and on the donor's scenery. donor_blanked is each donor's own blanked top-1, so
    follows_donor_without_silhouette leaves out swaps whose donor scenery alone gives the donor's class."""
    keeps, follows = swapped == labels, swapped == donor_labels
    known = donor_blanked != NO_PREDICTION
    silhouette = donor_blanked == donor_labels
    clean = known & ~silhouette
    return {"keeps_own_class": share(keeps), "keeps_own_class_ci": interval(lambda rows: share(keeps[rows]), groups),
            "follows_donor": share(follows), "follows_donor_ci": interval(lambda rows: share(follows[rows]), groups),
            "follows_donor_as_is": share(as_is == donor_labels),
            "keeps_own_among_right": share(keeps[as_is == labels]), "changed_prediction": share(swapped != as_is),
            "donor_silhouette_rate": share(silhouette[known]), "swaps_without_silhouette": int(clean.sum()),
            "follows_donor_without_silhouette": share(follows[clean])}


def verdict_change(before, after):
    """How often the verdict moves between two conditions on the same photos, and every move by name."""
    before, after = np.asarray(before), np.asarray(after)
    return {"photos": len(before), "changed": share(before != after),
            "transitions": {f"{old}->{new}": int(((before == old) & (after == new)).sum())
                            for old in VERDICTS for new in VERDICTS if old != new},
            "before": verdict_counts(before), "after": verdict_counts(after)}


def verdict_counts(labels):
    return {label: int((np.asarray(labels) == label).sum()) for label in VERDICTS}


def share(values):
    return float(np.mean(values)) if len(values) else math.nan


def interval(fn, groups):
    return list(grouped_bootstrap(fn, groups)) if len(groups) else NAN_INTERVAL


def flags(column):
    """A True/False column as booleans, also when it was read back from a CSV as text."""
    if column.dtype == bool:
        return column.to_numpy()
    mapped = column.astype("string").str.strip().str.lower().map({"true": True, "false": False})
    assert not mapped.isna().any(), f"{column.name} must hold True or False"
    return mapped.to_numpy(dtype=bool)

In [7]:
class ConditionPhotos(Dataset):
    """One condition's model inputs, built as the evaluation loader builds them, for `rows` (default all); `donors`
    gives each row's donor row (SWAPPED)."""

    def __init__(self, frame, condition, mode, size, donors=None, rows=None):
        self.frame, self.condition, self.mode, self.size = frame.reset_index(drop=True), condition, mode, size
        self.donors = donors
        self.rows = np.arange(len(self.frame)) if rows is None else np.asarray(rows)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        position = int(self.rows[index])
        row = self.frame.iloc[position]
        rgb, mask = self.photo(row, self.condition != AS_IS or self.mode in MASK_INPUT_MODES)
        if self.condition == BLANKED:
            rgb = blank_snake(rgb, mask)
        elif self.condition == SWAPPED:
            rgb = swap_background(rgb, mask, blank_snake(*self.photo(self.frame.iloc[self.donors[position]], True)))
        image, _ = apply_mode(rgb, mask if self.mode in MASK_INPUT_MODES else None, self.mode, row.get(BOX), self.size)
        return {"image": AUGMENT["eval"](image=image)["image"]}

    def photo(self, row, with_mask):
        rgb = load_rgb(row["image_file"])
        return rgb, load_mask(row["mask_file"], rgb.shape[:2]) if with_mask else None


def read_inputs(data_root, subset_csv, preds_root=None):
    metadata = pd.read_csv(Path(data_root, METADATA_FILE), dtype=ID_TYPES, low_memory=False)
    preds = None if preds_root is None else pd.read_csv(Path(preds_root, FRONTEND_PREDS_FILE))
    return Tables(metadata, pd.read_csv(Path(data_root, CLASSES_FILE)), pd.read_csv(subset_csv, dtype=ID_TYPES), None,
                  preds)


def training_frame(tables, cfg):
    """The run's training photos as run_frames builds them."""
    metadata = tables.metadata
    rows = metadata[(metadata["split"] == "train") & metadata["path"].isin(tables.subset["path"])]
    train = cap_train(build_frame(rows, tables.classes, ["train"]))
    return train[~train["species"].isin(cfg.exclude_species)].reset_index(drop=True)


def validation_frame(tables, cfg, data_root, preds_root):
    """Full validation, less exclude_species, with the front end's predicted masks and boxes."""
    known = ~tables.metadata["species"].isin(cfg.exclude_species)
    frame = build_frame(tables.metadata[known], tables.classes, ["val"], frontend_preds=tables.frontend_preds,
                        mask_source=PREDICTED, preds_root=preds_root)
    return with_files(frame, data_root)


def mask_status(frame, workers=QUALITY_WORKERS):
    """Per photo: MASK_USABLE when its mask has the photo's size and survives cleaning, else MASK_NONE,
    MASK_UNREADABLE, MASK_MISFIT or MASK_DROPPED."""
    def status(path, shape):
        if not path:
            return MASK_NONE
        try:
            with Image.open(path) as image:
                if (image.height, image.width) != shape:
                    return MASK_MISFIT
        except PHOTO_DECODE_ERRORS:
            return MASK_UNREADABLE
        return MASK_USABLE if load_mask(path, shape) is not None else MASK_DROPPED

    files = frame["mask_file"].where(frame["mask_valid"], "")
    shapes = list(zip(frame["height"].astype(int), frame["width"].astype(int)))
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return np.array(list(pool.map(status, files, shapes)), dtype=object)


def init_opencv(_):
    cv2.setNumThreads(OPENCV_THREADS_PER_WORKER)


def score_run(run_dir, tables, data_root, preds_root, quality, out_dir, device=None, num_workers=0, limit=None,
              medical=None, few_photographers=FEW_PHOTOGRAPHERS_CUTOFF, sources=None, recall_bound=POINT):
    """R1 on one run: validation as is, blanked and swapped, the slices and the verdict changes, written to out_dir.
    recall_bound must be the one the run's decision file was fitted with."""
    started = time.perf_counter()
    device = torch.device(device) if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    run_dir, out_dir, preds_root = Path(run_dir), Path(out_dir), Path(preds_root)
    out_dir.mkdir(parents=True, exist_ok=True)
    cfg, model, epoch = load_checkpoint(run_dir, device)
    mode, taxa = FRONT_END_MODES[cfg.front_end], class_tables(tables.classes)
    medical = pd.read_csv(MEDICAL_CSV, keep_default_na=False) if medical is None else medical
    danger = danger_tables(tables.classes, medical)
    prototypes = np.load(run_dir / PROTOTYPES_FILE)[PROTOTYPE_SET]
    thresholds = fit_thresholds(dict(np.load(run_dir / CALIBRATION_OUTPUTS)), prototypes, danger,
                                recall_bound=recall_bound)
    summary = preds_root / SUMMARY_FILE
    segmenter = (json.loads(summary.read_text()).get("winners", {}).get(S1, NOT_AVAILABLE) if summary.exists()
                 else NOT_AVAILABLE)
    provenance = {"run_id": cfg.run_id, "run_dir": str(run_dir), "checkpoint_epoch": epoch,
                  "checkpoint_sha256": file_sha256(run_dir / CHECKPOINT_FILE), "preds_root": str(preds_root),
                  "frontend_preds_sha256": file_sha256(preds_root / FRONTEND_PREDS_FILE), "segmenter": segmenter,
                  **(sources or {})}
    train = training_frame(tables, cfg)
    val = validation_frame(tables, cfg, data_root, preds_root)
    val = (val if limit is None else val.head(limit)).reset_index(drop=True)
    few = few_photographer_classes(train, taxa.n_classes, few_photographers)
    rows = quality.set_index("path").loc[val["path"]]
    val = val.assign(dark=flags(rows["dark"]), blurry=flags(rows["blurry"]),
                     few_photographers=few[val["class_id"].to_numpy()])
    warmed = warm_page_cache([*val["image_file"], *val.loc[val["mask_valid"], "mask_file"]])
    status = mask_status(val)
    if not (status == MASK_USABLE).any():
        raise ValueError(f"no validation photo has a usable predicted mask in {preds_root}, so nothing can be "
                         f"blanked or swapped; check frontend_preds.csv")

    def run(name, rows, donors=None):
        """name's outputs on val's `rows`, with their verdicts, also saved to OUTPUT_FILES[name]."""
        photos = ConditionPhotos(val, name, mode, cfg.input_size, donors, rows)
        loader = DataLoader(photos, batch_size=cfg.batch_size * EVAL_BATCH_SCALE, shuffle=False,
                            num_workers=num_workers, pin_memory=device.type == "cuda", worker_init_fn=init_opencv)
        outputs = predict(model, loader, device, amp=device.type == "cuda")
        outputs["verdict"] = decide(outputs, thresholds, prototypes, danger).label
        kept, saved = val.iloc[rows].reset_index(drop=True), {**outputs}
        if donors is not None:
            kept = kept.assign(donor_path=val["path"].to_numpy()[donors[rows]],
                               donor_class_id=val["class_id"].to_numpy()[donors[rows]])
        if "embedding" in saved:
            saved["embedding"] = saved["embedding"].astype(np.float16)
        save_outputs(out_dir / OUTPUT_FILES[name], kept, saved, () if donors is None else ("donor_path", "donor_class_id"))
        return outputs

    rows = {AS_IS: np.arange(len(val))}
    outputs = {AS_IS: run(AS_IS, rows[AS_IS])}
    check = instrument_check(run_dir / VAL_OUTPUTS, val, outputs[AS_IS])
    if check != NOT_AVAILABLE and not check["top1_agreement"] >= INSTRUMENT_FLOOR:
        raise RuntimeError(f"instrument check failed: the as-is pass gives the run's own saved top-1 on "
                           f"{check['top1_agreement']} of {check['photos']} photos, below {INSTRUMENT_FLOOR}, so it "
                           f"is not the run's model on the run's inputs; (b) and (c) were not run")
    usable = status == MASK_USABLE
    donors = draw_donors(val["class_id"].to_numpy(), usable, seed=SWAP_SEED)
    rows[BLANKED], rows[SWAPPED] = np.flatnonzero(usable), np.flatnonzero(donors != NO_DONOR)
    outputs[BLANKED] = run(BLANKED, rows[BLANKED])
    outputs[SWAPPED] = run(SWAPPED, rows[SWAPPED], donors)

    labels, groups = val["class_id"].to_numpy(), photographer_groups(val)
    probs = {name: softmax(outputs[name]["species_logits"]) for name in CONDITIONS}
    predicted = {name: probs[name].argmax(axis=1) for name in CONDITIONS}
    blanked_top1 = np.full(len(val), NO_PREDICTION)
    blanked_top1[rows[BLANKED]] = predicted[BLANKED]
    train_counts = np.bincount(train["class_id"], minlength=taxa.n_classes)
    val_counts = np.bincount(labels, minlength=taxa.n_classes)

    def scored(name, members=None):
        members = np.arange(len(rows[name])) if members is None else members
        return scores(labels[rows[name]][members], probs[name][members], groups[rows[name]][members], train_counts,
                      val_counts, taxa.tier)

    def by_india(report, members, name):
        return {name: report(members), "by_india": {"india": report(members[india[members]]),
                                                    "not_india": report(members[~india[members]])}}

    india = flags(val["in_india"])
    misfit = status == MASK_MISFIT
    metrics = {
        "run": cfg.run_id, "front_end": cfg.front_end, "input_mode": mode, "checkpoint_epoch": epoch,
        "masks": MASKS_NOTE, "provenance": provenance, "resumed": [],
        "photos": {"val": len(val), "scored": len(val), "with_mask": int(usable.sum()),
                   "without_mask": int((~usable).sum()), "mask_dropped_by_cleaning": int((status == MASK_DROPPED).sum()),
                   "mask_size_mismatch": int(misfit.sum()),
                   "mask_size_mismatch_examples": val.loc[misfit, "path"].head(ERROR_EXAMPLES).tolist(),
                   "mask_unreadable": int((status == MASK_UNREADABLE).sum()), "blanked": len(rows[BLANKED]),
                   "swapped": len(rows[SWAPPED]), "without_donor": int(usable.sum()) - len(rows[SWAPPED]),
                   "errors": {**dict.fromkeys(CONDITIONS, 0), "examples": []}},
        "instrument_check": check,
        "conditions": {
            AS_IS: scored(AS_IS), "as_is_masked": scored(AS_IS, np.flatnonzero(usable)),
            BLANKED: {**scored(BLANKED), **by_india(
                lambda members: shortcut_report(labels[members], predicted[AS_IS][members],
                                                predicted[BLANKED][np.searchsorted(rows[BLANKED], members)],
                                                groups[members]),
                rows[BLANKED], "shortcut"), "caveats": [MASK_MISS_NOTE, SHORTCUT_SILHOUETTE_NOTE]},
            SWAPPED: {**scored(SWAPPED), "as_is_same_photos": scored(AS_IS, rows[SWAPPED]), **by_india(
                lambda members: donor_report(labels[members], labels[donors[members]], predicted[AS_IS][members],
                                             predicted[SWAPPED][np.searchsorted(rows[SWAPPED], members)],
                                             groups[members], blanked_top1[donors[members]]),
                rows[SWAPPED], "donor"), "caveats": [MASK_MISS_NOTE, DONOR_SILHOUETTE_NOTE]},
        },
        "verdicts": {"note": VERDICT_NOTE, "recall_bound": recall_bound, "thresholds": asdict(thresholds),
                     AS_IS: verdict_counts(outputs[AS_IS]["verdict"]),
                     **{name: verdict_change(outputs[AS_IS]["verdict"][rows[name]], outputs[name]["verdict"])
                        for name in (BLANKED, SWAPPED)}},
        "slices": slice_metrics(val, probs[AS_IS], train_counts=train_counts, tier_of_class=taxa.tier,
                                val_counts=val_counts),
        "quality": quality_summary(quality),
        "settings": {"fill_rgb": BLANK_RGB, "dark_below": DARK_BELOW, "blur_below": BLUR_BELOW, "swap_seed": SWAP_SEED,
                     "donor": DONOR_NOTE, "few_photographers_at_most": few_photographers,
                     "few_photographer_classes": int(few.sum()), "shortcut": SHORTCUT_NOTE,
                     "prototype_set": PROTOTYPE_SET, "intervals": BOOTSTRAP_NOTE, "limit": limit,
                     "instrument_floor": INSTRUMENT_FLOOR},
        "warm_cache": warmed, "subset_hash": subset_hash(tables.subset["path"]), "versions": library_versions(device),
        "seconds": time.perf_counter() - started,
    }
    return write_json(out_dir / METRICS_FILE, metrics)


def instrument_check(saved_file, frame, outputs):
    """The as-is pass against the run's saved validation outputs, photo by photo: the same top-1, AMP rounding aside."""
    if not Path(saved_file).exists():
        return NOT_AVAILABLE
    with np.load(saved_file) as saved:
        position = {path: index for index, path in enumerate(saved["paths"])}
        pairs = np.array([(row, position[path]) for row, path in enumerate(frame["path"]) if path in position],
                         dtype=int).reshape(-1, 2)
        theirs = saved["species_logits"][pairs[:, 1]]
    mine = outputs["species_logits"][pairs[:, 0]]
    return {"photos": len(pairs), "top1_agreement": share(mine.argmax(axis=1) == theirs.argmax(axis=1)),
            "max_abs_logit_diff": float(np.abs(mine - theirs).max()) if len(pairs) else math.nan}


def write_quality(data_root, subset_csv, out_dir, workers=QUALITY_WORKERS, limit=None):
    """Writes quality.csv for validation, calibration and unknown-pool half A; returns the line to print."""
    tables = read_inputs(data_root, subset_csv)
    frames = {name: frame if limit is None else frame.head(limit)
              for name, frame in quality_frames(tables.metadata, tables.subset).items()}
    warmed = warm_page_cache(Path(data_root, IMAGE_FOLDER, path) for frame in frames.values() for path in frame["path"])
    quality = quality_table(frames, data_root, workers)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    quality.to_csv(Path(out_dir) / QUALITY_FILE, index=False)
    return f"{QUALITY_FILE}: {len(quality):,} photos, page cache {warmed}; {quality_summary(quality)}"


def score_line(metrics):
    conditions = metrics["conditions"]
    return (f"{metrics['run']}: photos {metrics['photos']}; as is {conditions[AS_IS]['top1']:.3f} top-1; "
            f"shortcut rate {conditions[BLANKED]['shortcut']['shortcut_rate']:.3f}; follows donor "
            f"{conditions[SWAPPED]['donor']['follows_donor']:.3f}")

## 1. Page cache, then the quality table

A cold read on the dataset mount takes about 60 ms; reading every photo and predicted mask once, 64 at a time, puts
them in the page cache that every process shares. Then the CPU measures every validation, calibration and
unknown-pool half A photo as stored (before the exposure fix) into `quality.csv`.

In [8]:
from serpentguard.data import IMAGE_FOLDER
from serpentguard.run_data import WARM_THREADS, warm_page_cache

tables = read_inputs(DATA, SUBSET, PREDS)
photos = [DATA / IMAGE_FOLDER / path for frame in quality_frames(tables.metadata, tables.subset).values()
          for path in frame["path"]]
masks = validation_frame(tables, cfg, DATA, PREDS).query("mask_valid")["mask_file"]
warmed = warm_page_cache([*photos, *masks], threads=WARM_THREADS)
print(f"page cache: {warmed['files']:,} files ({warmed['missing']} missing), {warmed['mb']:,.0f} MB in "
      f"{warmed['seconds']:.0f} s")

page cache: 41,578 files (0 missing), 3,569 MB in 114 s


In [9]:
line = write_quality(DATA, SUBSET, OUT, workers=workers, limit=LIMIT)
(OUT / "quality.log").write_text(line + "\n")
print(line)

quality.csv: 25,601 photos, page cache {'files': 25601, 'missing': 0, 'seconds': 6.188014775999989, 'mb': 3536.6712}; {'val': {'photos': 17151, 'dark': 0.13363652265174042, 'blurry': 0.04087225234680194}, 'calibration': {'photos': 7999, 'dark': 0.06350793849231154, 'blurry': 0.04963120390048756}, 'unknown': {'photos': 451, 'dark': 0.11529933481152993, 'blurry': 0.015521064301552107}}


## 2. Score the run on the GPU

The run's `robustness_metrics.json` records provenance (the run and predictions kernels, the checkpoint's and
`frontend_preds.csv`'s SHA-256, the segmenter), the photos with and without a usable predicted mask, an instrument check (the as-is pass against the run's own saved validation outputs,
photo by photo; below `INSTRUMENT_FLOOR` the run stops before (b) and (c)), the three conditions with their caveats,
the verdict changes, the slices, the quality shares, the settings, the page-cache warm-up, versions and the
subset hash.

In [10]:
RUN_OUT = OUT / RUN_ID
metrics = score_run(RUN, read_inputs(DATA, SUBSET, PREDS), DATA, PREDS, pd.read_csv(OUT / QUALITY_FILE), RUN_OUT,
                    device="cuda:0", num_workers=workers, limit=LIMIT,
                    few_photographers=FEW_PHOTOGRAPHERS_CUTOFF if FEW_PHOTOGRAPHERS is None else FEW_PHOTOGRAPHERS,
                    sources={"run_kernel": RUN_KERNEL, "preds_kernel": PREDS_KERNEL}, recall_bound=RECALL_BOUND)
line = score_line(metrics)
(RUN_OUT / "robustness.log").write_text(line + "\n")
print(line)

H1_top1_s1: photos {'val': 17151, 'scored': 17151, 'with_mask': 15977, 'without_mask': 1174, 'mask_dropped_by_cleaning': 0, 'mask_size_mismatch': 0, 'mask_size_mismatch_examples': [], 'mask_unreadable': 0, 'blanked': 15977, 'swapped': 15977, 'without_donor': 0, 'errors': {'as_is': 0, 'blanked': 0, 'swapped': 0, 'examples': []}}; as is 0.622 top-1; shortcut rate 0.178; follows donor 0.024


## 3. Results

Shortcut rate and donor-following are on the photos with a usable predicted mask; the control column is how
often the photo was already called the donor's class before the swap, and the silhouette rows are the control for the
donor's blanked snake. Both rates are upper bounds where masks miss part of the snake. Slice gaps are top-1 inside the
slice minus outside it.

In [11]:
def interval(values):
    return f"[{values[0]:.3f}, {values[1]:.3f}]"


quality = pd.read_csv(OUT / QUALITY_FILE)
assert quality["path"].is_unique and set(quality["set"]) == {"val", "calibration", "unknown"}, "quality.csv sets"
assert not quality["path"].str.contains(r"(?:^|/)test(?:_extra)?/").any(), "quality.csv holds a test photo"
metrics = json.loads((RUN_OUT / METRICS_FILE).read_text())
assert metrics["subset_hash"].startswith(EXPECTED_SUBSET_HASH), f"{RUN_ID} read another subset"
check = metrics["instrument_check"]
assert check["top1_agreement"] >= INSTRUMENT_FLOOR, f"{RUN_ID}: the as-is pass disagrees with the run: {check}"
conditions, verdicts, slices = metrics["conditions"], metrics["verdicts"], metrics["slices"]
shortcut, donor = conditions["blanked"]["shortcut"], conditions["swapped"]["donor"]
photos = metrics["photos"]
summary = pd.Series({
    "run kernel": RUN_KERNEL, "run": RUN_ID, "front end": metrics["front_end"],
    "segmenter": metrics["provenance"]["segmenter"], "val photos": metrics["photos"]["val"],
    "photos dropped (as is / blanked / swapped)": " / ".join(str(photos["errors"][name])
                                                         for name in ("as_is", "blanked", "swapped")),
    "with mask": photos["with_mask"], "without mask": photos["without_mask"],
    "masks dropped by cleaning / wrong size / unreadable": " / ".join(
        str(photos[key]) for key in ("mask_dropped_by_cleaning", "mask_size_mismatch", "mask_unreadable")),
    "top-1 as is": conditions["as_is"]["top1"], "macro-F1 as is": conditions["as_is"]["macro_f1"],
    "top-1 as is (masked)": conditions["as_is_masked"]["top1"], "top-1 blanked": conditions["blanked"]["top1"],
    "macro-F1 blanked": conditions["blanked"]["macro_f1"], "shortcut rate": shortcut["shortcut_rate"],
    "shortcut 95% CI": interval(shortcut["shortcut_rate_ci"]),
    "shortcut India / not": f"{conditions['blanked']['by_india']['india']['shortcut_rate']:.3f} / "
                            f"{conditions['blanked']['by_india']['not_india']['shortcut_rate']:.3f}",
    "top-1 swapped": conditions["swapped"]["top1"], "keeps own class": donor["keeps_own_class"],
    "follows donor": donor["follows_donor"], "donor class as is (control)": donor["follows_donor_as_is"],
    "donor silhouette rate": donor["donor_silhouette_rate"],
    "follows donor without silhouette": donor["follows_donor_without_silhouette"],
    "verdict changed, blanked": verdicts["blanked"]["changed"],
    "dangerous -> not listed, blanked": verdicts["blanked"]["transitions"]["dangerous->not_listed"],
    "unknown when blanked": verdicts["blanked"]["after"]["unknown"] / verdicts["blanked"]["photos"],
    "verdict changed, swapped": verdicts["swapped"]["changed"],
    **{f"{name} gap": slices[name]["top1_gap"] for name in slices},
    **{f"{name} photos in / out": f"{slices[name]['in']['photos']} / {slices[name]['out']['photos']}"
       for name in slices},
    "few-photographer classes (cut-off)": f"{metrics['settings']['few_photographer_classes']} "
                                          f"(<= {metrics['settings']['few_photographers_at_most']})",
    "India gap 95% CI": interval(slices["india"]["top1_gap_ci"]),
    "instrument top-1 agreement": check["top1_agreement"], "read back": ", ".join(metrics["resumed"]) or "none",
    "minutes": metrics["seconds"] / 60})
summary.to_csv(RUN_OUT / "summary.csv", header=False)
print("quality flags (share of photos):")
print(quality.groupby("set")[["dark", "blurry"]].mean().round(3).to_string())
summary

quality flags (share of photos):
              dark  blurry
set                       
calibration  0.064   0.050
unknown      0.115   0.016
val          0.134   0.041


run kernel                                             dataset:puneetbajajiitm/serpentguard-m3-run-final
run                                                                                           H1_top1_s1
front end                                                                                             F1
segmenter                                              {'candidate': 'unet', 'kind': 'unet', 'weights...
val photos                                                                                         17151
photos dropped (as is / blanked / swapped)                                                     0 / 0 / 0
with mask                                                                                          15977
without mask                                                                                        1174
masks dropped by cleaning / wrong size / unreadable                                            0 / 0 / 0
top-1 as is                                            